# Stereo Depth Estimation

## 핵심 논문
- **RAFT-Stereo** (3DV 2021): RAFT optical flow 구조를 스테레오에 적용
- **PSMNet** (CVPR 2018): Pyramid Stereo Matching Network — 3D cost volume
- **SGM** (Hirschmuller 2008): Semi-Global Matching — classical SOTA

## 핵심 아이디어
카메라 2대(좌/우)로 같은 장면을 촬영 → 픽셀 시차(disparity)로 metric depth 계산
```
depth = focal_length × baseline / disparity
```
- LiDAR 없이 실제 거리(미터) 추정 가능
- Monocular depth(상대값)와 달리 scale-ambiguity 없음

## 구현 순서
1. 스테레오 카메라 기하학 (epipolar geometry)
2. 합성 스테레오 데이터 생성 (KITTI 포맷)
3. Classical: Block Matching (SAD/SSD)
4. Learning-based: Correlation Volume + Disparity Regression (PSMNet-lite)
5. 평가: EPE / D1 / 3PE
6. Classical vs Learning-based 비교

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import cv2
import time
from scipy.ndimage import gaussian_filter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. 스테레오 카메라 기하학

In [ ]:
# ── 스테레오 카메라 파라미터 (KITTI 기준) ──────────────────────────────
FOCAL_LENGTH = 721.5  # px
BASELINE     = 0.54   # m (좌우 카메라 간격)
CX, CY       = 609.6, 172.9  # principal point
IMG_H, IMG_W = 256, 512
MAX_DISP     = 192   # 탐색할 최대 disparity (px)

def disp_to_depth(disp, focal=FOCAL_LENGTH, baseline=BASELINE):
    """disparity(px) → depth(m). disp=0인 곳은 inf 처리."""
    depth = np.zeros_like(disp, dtype=np.float32)
    valid = disp > 0
    depth[valid] = focal * baseline / disp[valid]
    return depth

def depth_to_disp(depth, focal=FOCAL_LENGTH, baseline=BASELINE):
    disp = np.zeros_like(depth, dtype=np.float32)
    valid = depth > 0
    disp[valid] = focal * baseline / depth[valid]
    return disp

# 기하학 관계 시각화
depths = np.linspace(1, 80, 300)
disps  = FOCAL_LENGTH * BASELINE / depths

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(depths, disps, 'b-', linewidth=2)
axes[0].set_xlabel('Depth (m)')
axes[0].set_ylabel('Disparity (px)')
axes[0].set_title('Depth ↔ Disparity 관계 (비선형)')
axes[0].axvline(x=10, color='r', linestyle='--', alpha=0.5, label='10m')
axes[0].axvline(x=40, color='g', linestyle='--', alpha=0.5, label='40m')
axes[0].legend()
axes[0].grid(True)

# 거리별 1px disparity 오차가 depth에 미치는 영향
depth_err = FOCAL_LENGTH * BASELINE / disps**2  # dD/dd = -fB/d^2
axes[1].plot(depths, depth_err, 'r-', linewidth=2)
axes[1].set_xlabel('Depth (m)')
axes[1].set_ylabel('Depth error per 1px disp error (m)')
axes[1].set_title('1px disparity 오차 → depth 오차 (원거리일수록 급증)')
axes[1].grid(True)

plt.tight_layout()
plt.show()
print(f'10m 물체의 disparity: {FOCAL_LENGTH * BASELINE / 10:.1f}px')
print(f'40m 물체의 disparity: {FOCAL_LENGTH * BASELINE / 40:.1f}px')
print(f'=> 1px 오차가 40m에서 {FOCAL_LENGTH * BASELINE / 40**2:.2f}m 오차 유발')

## 2. 합성 스테레오 데이터 생성 (KITTI 포맷)

In [ ]:
class SyntheticStereoDataset(torch.utils.data.Dataset):
    """
    합성 스테레오 데이터셋.
    - 도로 + 차량 + 건물 장면을 레이어별로 렌더링
    - GT depth → GT disparity 계산
    - 좌 이미지 → 우 이미지: GT disp만큼 수평 이동 (에피폴라 제약)
    """
    def __init__(self, n_samples=500, h=IMG_H, w=IMG_W, max_disp=MAX_DISP):
        self.n = n_samples
        self.h, self.w = h, w
        self.max_disp = max_disp
        self.rng = np.random.default_rng(42)

    def _make_scene(self, seed):
        rng = np.random.default_rng(seed)
        img  = np.zeros((self.h, self.w, 3), dtype=np.float32)
        depth = np.ones((self.h, self.w), dtype=np.float32) * 80.0  # sky = 80m

        # 하늘 (그라디언트)
        sky_color = rng.uniform([0.4, 0.6, 0.7], [0.6, 0.8, 1.0])
        sky_h = int(self.h * 0.45)
        for c in range(3):
            img[:sky_h, :, c] = np.linspace(sky_color[c], sky_color[c]*0.7, sky_h)[:, None]

        # 도로 (원근감)
        road_color = rng.uniform([0.3, 0.3, 0.3], [0.5, 0.5, 0.5])
        road_h = self.h - sky_h
        for r in range(road_h):
            row = sky_h + r
            d = 2.0 + 60.0 * (1 - r / road_h)  # 가까울수록 r 큰 쪽
            depth[row, :] = d
            shade = 0.7 + 0.3 * (r / road_h)
            img[row, :] = road_color * shade

        # 차량 박스 3~6개
        n_cars = rng.integers(3, 7)
        for _ in range(n_cars):
            car_d  = rng.uniform(5, 50)
            car_h_px = int(self.h * 0.12 * (20 / car_d))
            car_w_px = int(self.w * 0.08 * (20 / car_d))
            car_h_px = max(8, min(car_h_px, self.h // 3))
            car_w_px = max(8, min(car_w_px, self.w // 4))

            y_base = sky_h + int(road_h * (1 - (car_d - 2) / 60))
            x_base = rng.integers(10, self.w - car_w_px - 10)
            y1 = max(sky_h, y_base - car_h_px)
            y2 = min(self.h, y_base)
            x1, x2 = x_base, x_base + car_w_px

            car_color = rng.uniform([0.1, 0.1, 0.1], [0.9, 0.9, 0.9])
            img[y1:y2, x1:x2] = car_color
            # 유리창
            win_h = max(2, (y2-y1)//3)
            img[y1:y1+win_h, x1+2:x2-2] = [0.6, 0.7, 0.8]
            depth[y1:y2, x1:x2] = car_d

        # 노이즈
        img += rng.normal(0, 0.01, img.shape)
        img = np.clip(img, 0, 1)
        return img, depth

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        img_left, depth_gt = self._make_scene(idx)
        disp_gt = depth_to_disp(depth_gt).astype(np.float32)
        disp_gt = np.clip(disp_gt, 0, self.max_disp - 1)

        # 우 이미지: 좌 이미지를 GT disparity만큼 수평 좌측으로 이동
        img_right = np.zeros_like(img_left)
        for r in range(self.h):
            for c in range(self.w):
                d = int(disp_gt[r, c])
                src_c = c + d  # 오른쪽 카메라에서의 열 위치
                if 0 <= src_c < self.w:
                    img_right[r, c] = img_left[r, src_c]

        # 벡터화 버전 (빠름)
        img_right = np.zeros_like(img_left)
        coords_c = np.arange(self.w)
        for r in range(self.h):
            src_c = coords_c + disp_gt[r].astype(int)
            valid = (src_c >= 0) & (src_c < self.w)
            img_right[r, valid] = img_left[r, src_c[valid]]

        left_t  = torch.from_numpy(img_left.transpose(2,0,1)).float()
        right_t = torch.from_numpy(img_right.transpose(2,0,1)).float()
        disp_t  = torch.from_numpy(disp_gt).float()
        return left_t, right_t, disp_t


# 데이터 확인
dataset = SyntheticStereoDataset(n_samples=600)
left, right, disp = dataset[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0,0].imshow(left.permute(1,2,0).numpy())
axes[0,0].set_title('좌 이미지 (Left)')
axes[0,1].imshow(right.permute(1,2,0).numpy())
axes[0,1].set_title('우 이미지 (Right — 좌측으로 시프트)')
im = axes[1,0].imshow(disp.numpy(), cmap='plasma')
axes[1,0].set_title('GT Disparity (px)')
plt.colorbar(im, ax=axes[1,0])
depth_vis = disp_to_depth(disp.numpy())
im2 = axes[1,1].imshow(depth_vis, cmap='turbo', vmin=0, vmax=80)
axes[1,1].set_title('GT Depth (m)')
plt.colorbar(im2, ax=axes[1,1])
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'데이터셋: {len(dataset)}개')
print(f'Disparity 범위: {disp.min():.1f} ~ {disp.max():.1f} px')
print(f'Depth 범위: {depth_vis[depth_vis>0].min():.1f} ~ {depth_vis.max():.1f} m')

## 3. Classical: Block Matching (SAD)

Sum of Absolute Differences: 좌 이미지의 각 픽셀 패치와 우 이미지의 에피폴라 라인 상 패치를 비교해 최소 SAD 위치 = disparity

In [ ]:
def block_matching_sad(left_np, right_np, max_disp=MAX_DISP, block_size=9):
    """
    Block Matching (SAD) — 스테레오 disparity 추정.
    left_np, right_np: (H, W, 3) float32 [0,1]
    반환: disp_map (H, W) float32
    """
    H, W = left_np.shape[:2]
    half = block_size // 2

    # 그레이스케일 변환
    left_g  = (left_np  @ [0.299, 0.587, 0.114]).astype(np.float32)
    right_g = (right_np @ [0.299, 0.587, 0.114]).astype(np.float32)

    disp_map = np.zeros((H, W), dtype=np.float32)

    for r in range(half, H - half):
        left_patch_row = np.lib.stride_tricks.sliding_window_view(
            left_g[r-half:r+half+1], (block_size, block_size)
        )  # (1, W-bs+1, bs, bs)

        for c in range(half, W - half):
            patch_l = left_g[r-half:r+half+1, c-half:c+half+1]
            best_sad = float('inf')
            best_d   = 0
            for d in range(max_disp):
                c_r = c + d
                if c_r + half >= W:
                    break
                patch_r = right_g[r-half:r+half+1, c_r-half:c_r+half+1]
                sad = np.abs(patch_l - patch_r).sum()
                if sad < best_sad:
                    best_sad = sad
                    best_d   = d
            disp_map[r, c] = best_d
    return disp_map


def block_matching_vectorized(left_np, right_np, max_disp=64, block_size=7):
    """
    벡터화 SAD Block Matching — 실용 속도.
    """
    left_g  = (left_np  @ [0.299, 0.587, 0.114]).astype(np.float32)
    right_g = (right_np @ [0.299, 0.587, 0.114]).astype(np.float32)
    H, W = left_g.shape
    half = block_size // 2

    # 적분 이미지로 빠른 패치 합산
    cost_volume = np.full((max_disp, H, W), np.inf, dtype=np.float32)

    for d in range(max_disp):
        # 우 이미지를 d만큼 오른쪽으로 시프트
        right_shifted = np.zeros_like(right_g)
        right_shifted[:, d:] = right_g[:, :W-d]
        diff = np.abs(left_g - right_shifted)
        # 박스 필터 (패치 합산)
        kernel = np.ones((block_size, block_size)) / block_size**2
        cost_volume[d] = cv2.filter2D(diff, -1, kernel)

    disp_map = np.argmin(cost_volume, axis=0).astype(np.float32)
    # 경계 마스킹
    disp_map[:half, :] = 0
    disp_map[-half:, :] = 0
    disp_map[:, :half] = 0
    disp_map[:, -half:] = 0
    return disp_map


# 테스트
left_np  = left.permute(1,2,0).numpy()
right_np = right.permute(1,2,0).numpy()
disp_gt  = disp.numpy()

t0 = time.time()
disp_bm = block_matching_vectorized(left_np, right_np, max_disp=MAX_DISP, block_size=7)
t_bm = time.time() - t0
print(f'Block Matching 소요: {t_bm:.2f}s')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(left_np)
axes[0].set_title('좌 이미지')
im1 = axes[1].imshow(disp_gt, cmap='plasma', vmin=0, vmax=MAX_DISP)
axes[1].set_title('GT Disparity')
plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(disp_bm, cmap='plasma', vmin=0, vmax=MAX_DISP)
axes[2].set_title(f'Block Matching (SAD)')
plt.colorbar(im2, ax=axes[2])
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 평가 메트릭: EPE / D1 / 3PE

In [ ]:
def stereo_metrics(pred_disp, gt_disp, max_disp=MAX_DISP, threshold=3.0):
    """
    KITTI Stereo 표준 메트릭.
    - EPE  : End-Point Error — 예측/GT disparity 평균 절대 오차 (px)
    - D1   : % pixels where |pred-gt| > 3px AND |pred-gt| > 5% of gt
    - 3PE  : % pixels where |pred-gt| > 3px (단순 버전)
    valid mask: gt > 0 and gt < max_disp
    """
    valid = (gt_disp > 0) & (gt_disp < max_disp)
    if valid.sum() == 0:
        return {'EPE': 0, 'D1': 0, '3PE': 0}

    pred = pred_disp[valid].astype(np.float32)
    gt   = gt_disp[valid].astype(np.float32)
    err  = np.abs(pred - gt)

    epe = err.mean()
    # D1: 절대 오차 > 3px AND 상대 오차 > 5%
    d1  = ((err > threshold) & (err > 0.05 * gt)).mean() * 100
    # 3PE: 절대 오차 > 3px
    pe3 = (err > threshold).mean() * 100
    return {'EPE': epe, 'D1': d1, '3PE': pe3}


bm_metrics = stereo_metrics(disp_bm, disp_gt)
print('=== Block Matching (SAD) 메트릭 ===')
print(f'  EPE : {bm_metrics["EPE"]:.3f} px')
print(f'  D1  : {bm_metrics["D1"]:.2f}%')
print(f'  3PE : {bm_metrics["3PE"]:.2f}%')

## 5. Learning-based: PSMNet-lite (Correlation Volume + Disparity Regression)

**PSMNet 핵심 아이디어:**
1. **Shared Feature Extractor**: 좌/우 이미지에서 동일 CNN으로 feature 추출
2. **Cost Volume**: 각 disparity d에 대해 left feature와 right feature(d만큼 시프트)를 concat → (D, H, W, C) 4D 텐서
3. **3D CNN**: cost volume을 3D conv로 처리 → 정규화된 cost
4. **Disparity Regression**: soft argmin으로 미분 가능한 예측값 계산

In [ ]:
# ── Feature Extractor ────────────────────────────────────────────────────
class FeatureExtractor(nn.Module):
    """좌/우 공유 feature extractor. (B,3,H,W) → (B,C,H/4,W/4)"""
    def __init__(self, out_ch=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, stride=1, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, out_ch, 3, stride=1, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


# ── Cost Volume 구성 ─────────────────────────────────────────────────────
def build_cost_volume(feat_left, feat_right, max_disp):
    """
    PSMNet 스타일 cost volume.
    feat_left, feat_right: (B, C, H, W) — 1/4 해상도
    반환: cost_vol (B, 2C, max_disp//4, H, W)

    각 disparity d에 대해:
      - left  feature: 그대로 (전체 W)
      - right feature: d만큼 오른쪽으로 시프트 (왼쪽 d열 = 0 패딩)
    """
    B, C, H, W = feat_left.shape
    D = max_disp // 4  # feature는 1/4 해상도이므로 disp도 1/4
    cost = torch.zeros(B, 2*C, D, H, W, device=feat_left.device)
    for d in range(D):
        cost[:, :C, d] = feat_left          # left: 항상 전체 W 그대로
        if d == 0:
            cost[:, C:, d] = feat_right
        else:
            # right를 d만큼 오른쪽 시프트: 열 d~W-1 에 right의 열 0~W-d-1 할당
            cost[:, C:, d, :, d:] = feat_right[:, :, :, :W-d]
    return cost  # (B, 2C, D, H, W)


# ── 3D Cost Aggregation ──────────────────────────────────────────────────
class CostAggregation(nn.Module):
    """(B, 2C, D, H, W) → (B, 1, D, H, W) 정규화된 cost."""
    def __init__(self, in_ch=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, 32, 3, padding=1), nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            nn.Conv3d(32, 32, 3, padding=1),    nn.BatchNorm3d(32), nn.ReLU(inplace=True),
            nn.Conv3d(32, 16, 3, padding=1),    nn.BatchNorm3d(16), nn.ReLU(inplace=True),
            nn.Conv3d(16, 1,  3, padding=1),
        )  # (B, 1, D, H, W)
    def forward(self, cost):
        return self.net(cost)


# ── Disparity Regression (soft argmin) ──────────────────────────────────
def disparity_regression(cost_logit, max_disp):
    """
    Soft argmin: d_hat = sum(d * softmax(-cost)[d])
    미분 가능 → gradient flow 유지
    cost_logit: (B, 1, D, H, W)
    반환: disp (B, H, W) — 1/4 해상도
    """
    B, _, D, H, W = cost_logit.shape
    cost_sq = cost_logit.squeeze(1)  # (B, D, H, W)
    prob    = F.softmax(-cost_sq, dim=1)  # (B, D, H, W)
    # disparity index 0~D-1 → 실제 disp 0~max_disp
    d_vals  = torch.arange(D, device=cost_logit.device).float() * 4  # 1/4 스케일 보정
    d_vals  = d_vals.view(1, D, 1, 1).expand(B, D, H, W)
    disp    = (prob * d_vals).sum(dim=1)  # (B, H, W)
    return disp


# ── 전체 PSMNet-lite ─────────────────────────────────────────────────────
class PSMNetLite(nn.Module):
    def __init__(self, max_disp=MAX_DISP, feat_ch=32):
        super().__init__()
        self.max_disp = max_disp
        self.feature  = FeatureExtractor(out_ch=feat_ch)
        self.aggr     = CostAggregation(in_ch=feat_ch*2)

    def forward(self, left, right):
        fl = self.feature(left)
        fr = self.feature(right)
        cost = build_cost_volume(fl, fr, self.max_disp)  # (B,2C,D,H,W)
        cost_agg = self.aggr(cost)                        # (B,1,D,H,W)
        disp_low = disparity_regression(cost_agg, self.max_disp)  # (B,H/4,W/4)
        # 원본 해상도로 upsample
        disp_up = F.interpolate(disp_low.unsqueeze(1),
                                scale_factor=4,
                                mode='bilinear',
                                align_corners=False).squeeze(1)  # (B,H,W)
        return disp_up


model = PSMNetLite(max_disp=MAX_DISP).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'PSMNet-lite 파라미터: {total_params/1e6:.2f}M')

# 빠른 forward 테스트
dummy_l = torch.randn(2, 3, IMG_H, IMG_W).to(device)
dummy_r = torch.randn(2, 3, IMG_H, IMG_W).to(device)
with torch.no_grad():
    out = model(dummy_l, dummy_r)
print(f'출력 shape: {out.shape}')  # (2, H, W)

## 6. 학습

In [ ]:
from torch.utils.data import DataLoader, random_split

# 데이터 분할
dataset = SyntheticStereoDataset(n_samples=600)
n_val = 100
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

# Smooth L1 loss (disparity regression 표준)
criterion = nn.SmoothL1Loss(beta=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 25
history = {'train_loss': [], 'val_epe': [], 'val_d1': []}

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for left_b, right_b, disp_b in train_loader:
        left_b  = left_b.to(device)
        right_b = right_b.to(device)
        disp_b  = disp_b.to(device)

        pred = model(left_b, right_b)
        valid_mask = (disp_b > 0) & (disp_b < MAX_DISP)
        loss = criterion(pred[valid_mask], disp_b[valid_mask])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    # ── Validation ──
    model.eval()
    epe_list, d1_list = [], []
    with torch.no_grad():
        for left_b, right_b, disp_b in val_loader:
            pred = model(left_b.to(device), right_b.to(device)).cpu().numpy()
            gt   = disp_b.numpy()
            for p, g in zip(pred, gt):
                m = stereo_metrics(p, g)
                epe_list.append(m['EPE'])
                d1_list.append(m['D1'])

    val_epe = np.mean(epe_list)
    val_d1  = np.mean(d1_list)
    history['train_loss'].append(train_loss)
    history['val_epe'].append(val_epe)
    history['val_d1'].append(val_d1)

    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss:.4f} | '
              f'Val EPE: {val_epe:.3f}px | D1: {val_d1:.2f}%')

print('\n학습 완료!')

## 7. 학습 곡선 시각화

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], 'b-o', markersize=3)
axes[0].set_title('Train Loss (Smooth L1)')
axes[0].set_xlabel('Epoch')
axes[0].grid(True)

axes[1].plot(history['val_epe'], 'r-o', markersize=3)
axes[1].set_title('Val EPE (px) — End-Point Error')
axes[1].set_xlabel('Epoch')
axes[1].grid(True)

axes[2].plot(history['val_d1'], 'g-o', markersize=3)
axes[2].set_title('Val D1 (%) — 3px 초과 픽셀 비율')
axes[2].set_xlabel('Epoch')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 8. Classical vs Learning-based 비교

In [ ]:
# val 전체 BM 평가
bm_epe_list, bm_d1_list, bm_3pe_list = [], [], []
nn_epe_list, nn_d1_list, nn_3pe_list = [], [], []

model.eval()
with torch.no_grad():
    for left_b, right_b, disp_b in val_loader:
        # Learning-based
        pred_nn = model(left_b.to(device), right_b.to(device)).cpu().numpy()
        gt_np   = disp_b.numpy()
        for i in range(len(gt_np)):
            m = stereo_metrics(pred_nn[i], gt_np[i])
            nn_epe_list.append(m['EPE'])
            nn_d1_list.append(m['D1'])
            nn_3pe_list.append(m['3PE'])

            # Classical
            l_np = left_b[i].permute(1,2,0).numpy()
            r_np = right_b[i].permute(1,2,0).numpy()
            bm_pred = block_matching_vectorized(l_np, r_np, max_disp=MAX_DISP)
            m_bm = stereo_metrics(bm_pred, gt_np[i])
            bm_epe_list.append(m_bm['EPE'])
            bm_d1_list.append(m_bm['D1'])
            bm_3pe_list.append(m_bm['3PE'])

print('='*55)
print('          Block Matching    PSMNet-lite    개선')
print('='*55)
bm_epe_m = np.mean(bm_epe_list)
nn_epe_m = np.mean(nn_epe_list)
bm_d1_m  = np.mean(bm_d1_list)
nn_d1_m  = np.mean(nn_d1_list)
bm_3pe_m = np.mean(bm_3pe_list)
nn_3pe_m = np.mean(nn_3pe_list)
print(f'EPE (px)  {bm_epe_m:10.3f}    {nn_epe_m:10.3f}   {(bm_epe_m-nn_epe_m)/bm_epe_m*100:+.1f}%')
print(f'D1   (%)  {bm_d1_m:10.2f}    {nn_d1_m:10.2f}   {(bm_d1_m-nn_d1_m)/max(bm_d1_m,1e-6)*100:+.1f}%')
print(f'3PE  (%)  {bm_3pe_m:10.2f}    {nn_3pe_m:10.2f}   {(bm_3pe_m-nn_3pe_m)/max(bm_3pe_m,1e-6)*100:+.1f}%')
print('='*55)

## 9. 정성적 결과 비교

In [ ]:
# 샘플 3개 시각화
fig, axes = plt.subplots(3, 5, figsize=(20, 11))
cols = ['좌 이미지', 'GT Disparity', 'Block Matching', 'PSMNet-lite', 'Error Map (PSMNet)']
for ax, c in zip(axes[0], cols):
    ax.set_title(c, fontsize=10)

model.eval()
samples = [val_ds[i] for i in [0, 10, 20]]
with torch.no_grad():
    for row_idx, (left_s, right_s, disp_gt_s) in enumerate(samples):
        l_np = left_s.permute(1,2,0).numpy()
        r_np = right_s.permute(1,2,0).numpy()
        gt_s = disp_gt_s.numpy()

        pred_nn_s = model(left_s.unsqueeze(0).to(device),
                         right_s.unsqueeze(0).to(device)).squeeze(0).cpu().numpy()
        pred_bm_s = block_matching_vectorized(l_np, r_np, max_disp=MAX_DISP)

        axes[row_idx, 0].imshow(l_np)
        axes[row_idx, 1].imshow(gt_s, cmap='plasma', vmin=0, vmax=MAX_DISP)
        axes[row_idx, 2].imshow(pred_bm_s, cmap='plasma', vmin=0, vmax=MAX_DISP)
        axes[row_idx, 3].imshow(pred_nn_s, cmap='plasma', vmin=0, vmax=MAX_DISP)
        err = np.abs(pred_nn_s - gt_s)
        axes[row_idx, 4].imshow(err, cmap='hot', vmin=0, vmax=20)

for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 10. 최종 결과 요약

In [ ]:
# Depth 오차 (meter 환산)
nn_depth_err_list = []
model.eval()
with torch.no_grad():
    for left_b, right_b, disp_b in val_loader:
        pred_d = model(left_b.to(device), right_b.to(device)).cpu().numpy()
        gt_d   = disp_b.numpy()
        for p, g in zip(pred_d, gt_d):
            valid = (g > 1) & (g < MAX_DISP - 1)
            if valid.sum() == 0:
                continue
            depth_pred = disp_to_depth(p[valid])
            depth_gt   = disp_to_depth(g[valid])
            nn_depth_err_list.append(np.abs(depth_pred - depth_gt).mean())

avg_depth_err = np.mean(nn_depth_err_list)

print('=' * 60)
print('  skillup_round7 / 01 Stereo Depth — 최종 결과')
print('=' * 60)
print(f'  모델:         PSMNet-lite (FeatureNet + 4D Cost Volume + SoftArgmin)')
print(f'  데이터:       합성 KITTI 포맷 스테레오 (train=500, val=100)')
print(f'  EPE:          {nn_epe_m:.3f} px  (Block Matching: {bm_epe_m:.3f} px)')
print(f'  D1:           {nn_d1_m:.2f}%     (Block Matching: {bm_d1_m:.2f}%)')
print(f'  3PE:          {nn_3pe_m:.2f}%    (Block Matching: {bm_3pe_m:.2f}%)')
print(f'  Depth MAE:    {avg_depth_err:.3f} m  (metric depth, LiDAR 없이)')
print()
print('  구현 핵심:')
print('    1. depth = f×B/d  — 스테레오 삼각측량 원리 직접 구현')
print('    2. 4D Cost Volume  — 각 disparity에서 left/right feature concat')
print('    3. Soft Argmin     — 미분 가능한 disparity regression')
print('    4. EPE/D1/3PE      — KITTI Stereo 표준 평가 파이프라인')
print('    5. Classical vs Learning-based 정량 비교')
print('=' * 60)